# 01 — Sistemas multi-agente

En este notebook implementamos un sistema multi-agente donde varios agentes especializados colaboran para completar una tarea compleja. Cada agente tiene un rol definido (investigador, redactor, revisor) y un orquestador coordina el flujo de trabajo.

**Complemento del tutorial:** `tutoriales/agentes-avanzados/01-multi-agente.md`

**Requisitos:** Necesitas una clave de API de Anthropic en la variable de entorno `ANTHROPIC_API_KEY`.

In [ ]:
# %pip install anthropic

import anthropic
import json
import os

cliente = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
MODELO = "claude-haiku-4-5-20251001"

## 1. Agentes especializados

Cada agente es una función que llama a Claude con un `system` prompt específico para su rol. El agente recibe el contexto acumulado y devuelve su contribución como texto.

In [ ]:
def agente_investigador(tema: str) -> str:
    """Agente especializado en investigar y recopilar información sobre un tema."""
    respuesta = cliente.messages.create(
        model=MODELO,
        max_tokens=600,
        system=(
            "Eres un investigador experto. Tu tarea es analizar un tema y proporcionar "
            "los hechos clave, datos relevantes y perspectivas principales. "
            "Sé conciso y objetivo. Organiza la información en puntos claros."
        ),
        messages=[
            {"role": "user", "content": f"Investiga el siguiente tema: {tema}"}
        ],
    )
    return respuesta.content[0].text


def agente_redactor(tema: str, investigacion: str) -> str:
    """Agente especializado en redactar contenido claro y bien estructurado."""
    respuesta = cliente.messages.create(
        model=MODELO,
        max_tokens=800,
        system=(
            "Eres un redactor profesional. Tu tarea es tomar una investigación "
            "y transformarla en un texto fluido, bien estructurado y accesible "
            "para el público general. Usa párrafos cortos y lenguaje claro."
        ),
        messages=[
            {
                "role": "user",
                "content": (
                    f"Tema: {tema}\n\n"
                    f"Investigación disponible:\n{investigacion}\n\n"
                    "Redacta un artículo claro y bien estructurado basándote en esta información."
                ),
            }
        ],
    )
    return respuesta.content[0].text


def agente_revisor(tema: str, borrador: str) -> str:
    """Agente especializado en revisar y mejorar el texto final."""
    respuesta = cliente.messages.create(
        model=MODELO,
        max_tokens=900,
        system=(
            "Eres un editor experto. Tu tarea es revisar un borrador de texto y mejorarlo. "
            "Corrige errores, mejora la claridad, elimina redundancias y asegura que "
            "el texto sea coherente y tenga un buen flujo. "
            "Devuelve el texto revisado completo."
        ),
        messages=[
            {
                "role": "user",
                "content": (
                    f"Tema: {tema}\n\n"
                    f"Borrador a revisar:\n{borrador}\n\n"
                    "Revisa y mejora este texto. Devuelve la versión final corregida."
                ),
            }
        ],
    )
    return respuesta.content[0].text


print("Agentes definidos: investigador, redactor, revisor")

## 2. Orquestador

El orquestador coordina a los tres agentes en secuencia, pasando la salida de cada uno como entrada al siguiente. Este patrón se llama **pipeline secuencial** y es el más sencillo de los patrones multi-agente.

In [ ]:
def orquestador(tema: str) -> dict:
    """
    Coordina los tres agentes en secuencia para producir un artículo completo.

    Flujo:
      1. Agente investigador  → recopila hechos clave
      2. Agente redactor      → transforma los hechos en texto fluido
      3. Agente revisor       → pule y mejora el texto final

    Returns:
        Diccionario con las salidas de cada etapa y el resultado final.
    """
    print(f"Procesando tema: '{tema}'")
    print("-" * 50)

    # Etapa 1: Investigación
    print("[1/3] Agente investigador trabajando...")
    investigacion = agente_investigador(tema)
    print("      ✓ Investigación completada")

    # Etapa 2: Redacción
    print("[2/3] Agente redactor trabajando...")
    borrador = agente_redactor(tema, investigacion)
    print("      ✓ Borrador completado")

    # Etapa 3: Revisión
    print("[3/3] Agente revisor trabajando...")
    texto_final = agente_revisor(tema, borrador)
    print("      ✓ Revisión completada")

    print("-" * 50)
    print("Pipeline completado.")

    return {
        "tema": tema,
        "investigacion": investigacion,
        "borrador": borrador,
        "texto_final": texto_final,
    }


print("Orquestador definido.")

## 3. Ejecución

Ejecutamos el pipeline multi-agente con un tema de ejemplo. El proceso tarda unos segundos porque realiza tres llamadas a la API en secuencia.

In [ ]:
resultado = orquestador("impacto de la IA en el empleo")

print("\n" + "=" * 60)
print("TEXTO FINAL")
print("=" * 60)
print(resultado["texto_final"])

print("\n" + "=" * 60)
print("INVESTIGACIÓN (salida del agente 1)")
print("=" * 60)
print(resultado["investigacion"])

## 4. Tu turno

Experimenta con el sistema modificando los siguientes elementos:

1. **Cambia el tema**: Prueba con otros temas como `"cambio climático"`, `"criptomonedas"` o `"inteligencia artificial en medicina"`.

2. **Modifica los system prompts**: Adapta el tono o el estilo de cada agente. Por ejemplo, haz que el redactor escriba para un público técnico o que el revisor use un estilo más formal.

3. **Añade un cuarto agente**: Por ejemplo, un `agente_traductor` que traduzca el texto final al inglés, o un `agente_resumidor` que genere un resumen ejecutivo de tres puntos.

4. **Implementa ejecución paralela**: Los agentes no siempre tienen que ejecutarse en secuencia. ¿Puedes hacer que el investigador y otro agente trabajen en paralelo usando `threading` o `asyncio`?

```python
# Punto de partida para experimentar:
mi_resultado = orquestador("TU TEMA AQUÍ")
print(mi_resultado["texto_final"])
```